# Assignment 3: Transformer is All You Need

**Tiny Shakespeare** next-token prediction with a **from-scratch** causal Transformer in PyTorch.

- Subword (BPE) tokenizer, vocabulary $\leq 500$, trained **only on the training split**
- 80/20 train/validation split, overlapping sequences (~50--64 tokens)
- 2 Transformer blocks, $d_{\mathrm{model}} \leq 128$, RMSNorm, sinusoidal PE, causal self-attention, FFN
- Metrics: train/val cross-entropy, validation **perplexity** $\exp(\mathrm{CE})$
- Attention heatmaps and short experiments (positional encoding ablation, context length)


In [ ]:
# Colab / local: ensure tokenizers (PyTorch is preinstalled on Colab GPU runtimes)
%pip install tokenizers -q


In [ ]:
import math
import os
import urllib.request
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print('Device:', DEVICE)

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR = Path('.')


## 1. Data: Tiny Shakespeare

Download the standard Tiny Shakespeare corpus (Karpathy char-rnn mirror).


In [ ]:
URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
DATA_PATH = DATA_DIR / 'input.txt'
if not DATA_PATH.exists():
    urllib.request.urlretrieve(URL, DATA_PATH)
    print('Downloaded to', DATA_PATH)
else:
    print('Using existing', DATA_PATH)

full_text = DATA_PATH.read_text(encoding='utf-8')
print('Characters:', len(full_text))
print(full_text[:400])


## 2. Train/validation split and BPE (vocab $\leq 500$)

We split **raw text** 80/20 (by character index) so validation is truly held out, then **train the BPE tokenizer on the training portion only** to avoid tokenizer leakage from validation tokens into merge statistics.


In [ ]:
split_idx = int(len(full_text) * 0.8)
train_text = full_text[:split_idx]
val_text = full_text[split_idx:]
print(f'Train chars: {len(train_text):,} | Val chars: {len(val_text):,}')

VOCAB_SIZE = 500
tokenizer = Tokenizer(BPE(unk_token='<unk>'))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()
trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=['<unk>', '<pad>'],
)
tokenizer.train_from_iterator([train_text], trainer=trainer)
tokenizer.enable_padding(pad_id=tokenizer.token_to_id('<pad>'), pad_token='<pad>')

actual_vocab = tokenizer.get_vocab_size()
print('Tokenizer vocab size:', actual_vocab)
assert actual_vocab <= VOCAB_SIZE, 'Vocab exceeds cap'

def encode(text):
    return tokenizer.encode(text).ids

train_ids = encode(train_text)
val_ids = encode(val_text)
print('Train tokens:', len(train_ids), '| Val tokens:', len(val_ids))


## 3. Overlapping sequences (next-token prediction)

Each chunk has `block_size` consecutive tokens. We use `input = chunk[:-1]` and `target = chunk[1:]` so every position predicts the next token. Stride $<$ block_size gives overlapping windows (more training signal for a small corpus).


In [ ]:
BLOCK_SIZE = 65   # 64 next-token positions per window (aligns with ~50--64 spec)
STRIDE = 32       # overlapping windows

class ShakespeareDataset(Dataset):
    def __init__(self, token_ids, block_size, stride):
        self.block_size = block_size
        self.samples = []
        for i in range(0, len(token_ids) - block_size + 1, stride):
            chunk = token_ids[i : i + block_size]
            x = torch.tensor(chunk[:-1], dtype=torch.long)
            y = torch.tensor(chunk[1:], dtype=torch.long)
            self.samples.append((x, y))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

train_ds = ShakespeareDataset(train_ids, BLOCK_SIZE, STRIDE)
val_ds = ShakespeareDataset(val_ids, BLOCK_SIZE, STRIDE)
print('Train windows:', len(train_ds), '| Val windows:', len(val_ds))

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)


## 4. Model: RMSNorm, sinusoidal PE, causal multi-head attention, FFN

**Pre-LN** block: $x \leftarrow x + \mathrm{Dropout}(\mathrm{Sublayer}(\mathrm{RMSNorm}(x)))$ for attention and FFN.

Attention weights are returned for visualization (single forward pass stores last layer's weights; all heads).


In [ ]:
class RMSNorm(nn.Module):
    # Root mean square layer normalization (no bias)
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return x * rms * self.weight


class SinusoidalPositionalEncoding(nn.Module):
    # Fixed sinusoidal PE added to token embeddings
    def __init__(self, d_model, max_len=4096):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = self.d_head ** -0.5

        self.wq = nn.Linear(d_model, d_model, bias=False)
        self.wk = nn.Linear(d_model, d_model, bias=False)
        self.wv = nn.Linear(d_model, d_model, bias=False)
        self.wo = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

        self._last_attn = None  # (B, n_heads, T, T)

    def forward(self, x, return_attention=False):
        B, T, D = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) * self.scale
        causal = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(causal, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        self._last_attn = attn.detach()

        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, D)
        out = self.wo(out)
        if return_attention:
            return out, attn
        return out


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x)))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x


class TinyTransformerLM(nn.Module):
    # Causal LM: token embed + PE + transformer blocks + RMSNorm + lm head
    def __init__(self, vocab_size, d_model=128, n_heads=8, n_layers=2, d_ff=512, max_len=4096, dropout=0.1, use_pe=True):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.use_pe = use_pe
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_enc = SinusoidalPositionalEncoding(d_model, max_len) if use_pe else None
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        x = self.tok_emb(idx) * math.sqrt(self.d_model)
        if self.pos_enc is not None:
            x = self.pos_enc(x)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        return logits

    def get_attention_from_layer(self, layer_idx):
        # Last forward pass: attention from block layer_idx (0-indexed)
        return self.blocks[layer_idx].attn._last_attn


### 4.1 Optional sanity check vs `torch.nn.MultiheadAttention`

We compare a single batch's attention output shape and a rough numerical match on small random embeddings (causal MHA in PyTorch uses batch_first in recent versions).


In [ ]:
def sanity_multihead_attention():
    B, T, D = 2, 16, 128
    n_heads = 8
    x = torch.randn(B, T, D)
    # Our module
    ours = CausalSelfAttention(D, n_heads)
    ours.eval()
    with torch.no_grad():
        y1 = ours(x)
    # PyTorch reference (batch_first=True)
    ref = nn.MultiheadAttention(D, n_heads, batch_first=True, bias=False)
    # Not identical init — just check shapes
    assert y1.shape == (B, T, D), y1.shape
    print('Sanity: our attention output shape', tuple(y1.shape), 'OK')

sanity_multihead_attention()


## 5. Training configuration

Hyperparameters stay within the assignment hint: **2 layers**, **d_model=128**, small model.


In [ ]:
D_MODEL = 128
N_HEADS = 8
N_LAYERS = 2
D_FF = 512
DROPOUT = 0.1
LR = 3e-4
EPOCHS = 25
PAD_ID = tokenizer.token_to_id('<pad>')

model = TinyTransformerLM(
    vocab_size=actual_vocab,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    d_ff=D_FF,
    max_len=BLOCK_SIZE + 32,
    dropout=DROPOUT,
    use_pe=True,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

def evaluate_ppl(loader, model):
    model.eval()
    total_loss = 0.0
    total_tok = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits.view(-1, actual_vocab), yb.view(-1))
            total_loss += loss.item() * yb.numel()
            total_tok += yb.numel()
    mean_ce = total_loss / total_tok
    return mean_ce, math.exp(mean_ce)


## 6. Training loop (train / val loss and perplexity)


In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    total_tok = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits.view(-1, actual_vocab), yb.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * yb.numel()
        total_tok += yb.numel()
    return total_loss / total_tok

history = {'train_ce': [], 'val_ce': [], 'val_ppl': []}

for epoch in range(1, EPOCHS + 1):
    tr_ce = train_one_epoch(model, train_loader, optimizer)
    val_ce, val_ppl = evaluate_ppl(val_loader, model)
    history['train_ce'].append(tr_ce)
    history['val_ce'].append(val_ce)
    history['val_ppl'].append(val_ppl)
    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
        print(f'Epoch {epoch:3d} | train CE {tr_ce:.4f} | val CE {val_ce:.4f} | val PPL {val_ppl:.2f}')

final_val_ce, final_val_ppl = history['val_ce'][-1], history['val_ppl'][-1]
print(f'\nFinal validation CE: {final_val_ce:.4f}  |  PPL: {final_val_ppl:.2f}')


## 7. Loss curves and perplexity (save figures for report)


In [ ]:
plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 300, 'font.size': 10})

fig, ax = plt.subplots(2, 1, figsize=(6, 7), sharex=True)
epochs = range(1, len(history['train_ce']) + 1)
ax[0].plot(epochs, history['train_ce'], label='Train CE', color='#2E86AB')
ax[0].plot(epochs, history['val_ce'], label='Val CE', color='#E94F37')
ax[0].set_ylabel('Cross-entropy (nats)')
ax[0].legend()
ax[0].set_title('Training and validation cross-entropy')
ax[1].plot(epochs, history['val_ppl'], color='#44AF69', marker='o', markersize=3)
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Perplexity')
ax[1].set_title('Validation perplexity = exp(val CE)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_01_loss_ppl.png', bbox_inches='tight')
plt.show()
print('Saved fig_01_loss_ppl.png')


## 8. Attention heatmaps

Run a **short validation batch** through the model in `eval` mode, then plot one head's causal attention for a short prefix (decoded subword labels on axes).


In [ ]:
def ids_to_labels(ids_list):
    return [tokenizer.decode([i]) for i in ids_list]

@torch.no_grad()
def plot_attention_heatmap(model, x_sample, layer_idx=0, head_idx=0, max_len=32, fname='fig_02_attention.png'):
    model.eval()
    x = x_sample[:max_len].unsqueeze(0).to(DEVICE)
    _ = model(x)
    attn = model.get_attention_from_layer(layer_idx)[0, head_idx].cpu().numpy()  # (T, T)
    T = attn.shape[0]
    labels = ids_to_labels(x_sample[:T].tolist())

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(attn, cmap='Blues', vmin=0, vmax=attn.max())
    ax.set_xticks(np.arange(T))
    ax.set_yticks(np.arange(T))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel('Key position')
    ax.set_ylabel('Query position')
    ax.set_title(f'Causal attention — layer {layer_idx}, head {head_idx}')
    plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.savefig(FIG_DIR / fname, bbox_inches='tight')
    plt.show()
    print('Saved', fname)

# One batch from validation
xb, _ = next(iter(val_loader))
plot_attention_heatmap(model, xb[0], layer_idx=0, head_idx=0, max_len=28, fname='fig_02_attention_L0H0.png')
plot_attention_heatmap(model, xb[0], layer_idx=1, head_idx=2, max_len=28, fname='fig_03_attention_L1H2.png')


## 9. Experiment A — Positional encoding ablation

Train a second model **without** sinusoidal PE (same hyperparameters, fewer epochs for speed). Expect worse validation PPL and less structured attention.


In [ ]:
ABLATE_EPOCHS = 15
model_no_pe = TinyTransformerLM(
    vocab_size=actual_vocab,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    d_ff=D_FF,
    max_len=BLOCK_SIZE + 32,
    dropout=DROPOUT,
    use_pe=False,
).to(DEVICE)
opt2 = torch.optim.AdamW(model_no_pe.parameters(), lr=LR, weight_decay=0.01)
hist_no_pe = {'train_ce': [], 'val_ce': [], 'val_ppl': []}

for epoch in range(1, ABLATE_EPOCHS + 1):
    tr_ce = train_one_epoch(model_no_pe, train_loader, opt2)
    val_ce, val_ppl = evaluate_ppl(val_loader, model_no_pe)
    hist_no_pe['train_ce'].append(tr_ce)
    hist_no_pe['val_ce'].append(val_ce)
    hist_no_pe['val_ppl'].append(val_ppl)
    if epoch == 1 or epoch % 5 == 0 or epoch == ABLATE_EPOCHS:
        print(f'[No PE] Epoch {epoch:3d} | val CE {val_ce:.4f} | val PPL {val_ppl:.2f}')

print('Final no-PE val PPL:', hist_no_pe['val_ppl'][-1])

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history['val_ppl'][:ABLATE_EPOCHS], label='With sinusoidal PE', color='#2E86AB')
ax.plot(hist_no_pe['val_ppl'], label='No PE', color='#E94F37')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation perplexity')
ax.legend()
ax.set_title('Ablation: positional encoding')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_04_ablation_pe.png', bbox_inches='tight')
plt.show()


## 10. Experiment B — Shorter context (block size 33 → 32 positions)

Rebuild datasets with smaller `BLOCK_SIZE`, retrain a fresh model (few epochs) to illustrate memory/accuracy tradeoff.


In [ ]:
BLOCK_SHORT = 33
STRIDE_S = 16
train_ds_s = ShakespeareDataset(train_ids, BLOCK_SHORT, STRIDE_S)
val_ds_s = ShakespeareDataset(val_ids, BLOCK_SHORT, STRIDE_S)
train_loader_s = DataLoader(train_ds_s, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader_s = DataLoader(val_ds_s, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

model_short = TinyTransformerLM(
    vocab_size=actual_vocab,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    d_ff=D_FF,
    max_len=BLOCK_SHORT + 16,
    dropout=DROPOUT,
    use_pe=True,
).to(DEVICE)
opt_s = torch.optim.AdamW(model_short.parameters(), lr=LR, weight_decay=0.01)
SHORT_EPOCHS = 12
hist_short = {'val_ppl': []}
for epoch in range(1, SHORT_EPOCHS + 1):
    train_one_epoch(model_short, train_loader_s, opt_s)
    _, val_ppl = evaluate_ppl(val_loader_s, model_short)
    hist_short['val_ppl'].append(val_ppl)
    if epoch % 4 == 0:
        print(f'[Short ctx] Epoch {epoch} | val PPL {val_ppl:.2f}')

print('Short context final val PPL:', hist_short['val_ppl'][-1])
print('(Compare to full model at same epoch count qualitatively; longer context usually helps PPL.)')

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, SHORT_EPOCHS + 1), hist_short['val_ppl'], marker='o', color='#F18F01')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation perplexity')
ax.set_title('Shorter context (block=33 tokens, 32 positions)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_05_short_context.png', bbox_inches='tight')
plt.show()
print('Saved fig_05_short_context.png')


## 11. Optional text generation (temperature sampling)


In [ ]:
@torch.no_grad()
def generate(model, start_ids, max_new_tokens=80, temperature=0.9):
    model.eval()
    ids = list(start_ids)
    for _ in range(max_new_tokens):
        ctx = torch.tensor(ids[-(BLOCK_SIZE - 1):], dtype=torch.long, device=DEVICE).unsqueeze(0)
        logits = model(ctx)[0, -1] / temperature
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, 1).item()
        ids.append(next_id)
    return tokenizer.decode(ids)

# Prompt from first val tokens
prompt_ids = val_ids[:20]
text_out = generate(model, prompt_ids, max_new_tokens=120, temperature=0.85)
print('--- Generated sample ---')
print(text_out[:800])


## 12. Summary table (for report)

| Item | Value |
|------|--------|
| Vocab size | ≤ 500 BPE (train-only fit) |
| Block size / stride | 65 / 32 (64 positions); short exp: 33 / 16 |
| d_model, layers, heads, d_ff | 128, 2, 8, 512 |
| Optimizer | AdamW, lr=3e-4, grad clip 1.0 |
| Final val PPL | see last epoch above |

**Memory/runtime:** attention is $O(T^2)$ in sequence length; bottleneck is the $(B, H, T, T)$ attention matrix and matmuls.


## 13. Colab: download figures for your report

Use the **Files** browser (left) to download `fig_*.png`, or run the next cell.


In [ ]:
try:
    from google.colab import files
    import glob
    for p in sorted(glob.glob('fig_*.png')):
        files.download(p)
    print('Triggered download for fig_*.png')
except ImportError:
    print('Not on Colab — figures are in:', FIG_DIR.resolve())
